In [2]:
!pip install transformers datasets scikit-learn openai accelerate


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from datasets import Dataset
from scipy.special import softmax

In [4]:
train_df = pd.read_csv("/Users/nili/PycharmProjects/MasterProject/exist2025/Dataset/train.csv")
dev_df = pd.read_csv("/Users/nili/PycharmProjects/MasterProject/exist2025/Dataset/val.csv")
test_df = pd.read_csv("/Users/nili/PycharmProjects/MasterProject/exist2025/Dataset/test.csv")

print(train_df.head())
print(train_df.shape)

   test_case      id language  \
0  EXIST2023  101705       es   
1  EXIST2021   11108       es   
2  EXIST2021   10405       es   
3  EXIST2021    8658       en   
4  EXIST2023  200886       en   

                                                text  \
0  segundo: el mundo, y ecuador en especial, es u...   
1  @vox_es dicen que las mujeres son valientes pe...   
2  @elisacarrio jubilate, gorda termotanque de ca...   
3  you guys don’t understand the struggle of have...   
4  boris johnson never brushing his hair shows ge...   

                                        task1  \
0  ['YES', 'YES', 'YES', 'YES', 'YES', 'YES']   
1                                      sexist   
2                                      sexist   
3                                  non-sexist   
4     ['YES', 'YES', 'NO', 'NO', 'NO', 'YES']   

                                               task2  hard_label_task1  \
0  [['MISOGYNY-NON-SEXUAL-VIOLENCE'], ['STEREOTYP...                 1   
1                    

In [5]:
df = pd.concat([train_df, dev_df])

train_df, val_df = train_test_split(
    df,
    test_size=0.15,
    stratify=df["hard_label_task2"],
    random_state=42
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

In [6]:
import ast

def convert_to_dataset(df):

    df = df.copy()

    df["text"] = df["text"].fillna("").astype(str)

    # convert string "[0.0, ...]" → list
    df["soft_label_task2"] = df["soft_label_task2"].apply(ast.literal_eval)

    labels = np.array(df["soft_label_task2"].tolist(), dtype=np.float32)

    data = {
        "text": df["text"].tolist(),
        "labels": labels
    }

    return Dataset.from_dict(data)

train_dataset = convert_to_dataset(train_df)
val_dataset = convert_to_dataset(val_df)

In [7]:
#For test
test_dataset = Dataset.from_dict({
    "text": test_df["text"].tolist()
})

In [8]:
from transformers import AutoTokenizer
MODEL_NAME = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast= False)


In [9]:
MAX_LENGTH = 128

def tokenize(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )

train_dataset = train_dataset.map(tokenize)
val_dataset = val_dataset.map(tokenize)
test_dataset = test_dataset.map(tokenize)

Map:   0%|          | 0/13972 [00:00<?, ? examples/s]

Map:   0%|          | 0/2466 [00:00<?, ? examples/s]

Map:   0%|          | 0/1827 [00:00<?, ? examples/s]

In [10]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=6 

)

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
import torch.nn.functional as F

class SoftLabelTrainer(Trainer):

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):

        labels = inputs.pop("labels")

        outputs = model(**inputs)

        logits = outputs.logits

        log_probs = F.log_softmax(logits, dim=-1)

        loss = F.kl_div(
            log_probs,
            labels,
            reduction="batchmean"
        )

        return (loss, outputs) if return_outputs else loss

In [12]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    preds = np.argmax(logits, axis=1)

    true = np.argmax(labels, axis=1)

    return {
        "accuracy": accuracy_score(true, preds),
        "f1_macro": f1_score(true, preds, average="macro"),
        "f1_weighted": f1_score(true, preds, average="weighted")
    }

In [13]:
training_args = TrainingArguments(

    output_dir="./model_task2_soft",

    learning_rate=2e-5,
    eval_strategy= "epoch",
    save_strategy="epoch",

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=4,

    weight_decay=0.01,

    load_best_model_at_end=True,

    logging_dir="./logs",

    report_to=None
)

In [14]:
trainer = SoftLabelTrainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=val_dataset,

    tokenizer=tokenizer,

    compute_metrics=compute_metrics
)

/var/folders/w2/7wgmb9zn01d8k6wv97czc7th0000gn/T/ipykernel_16430/794875127.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SoftLabelTrainer.__init__`. Use `processing_class` instead.
  trainer = SoftLabelTrainer(


In [15]:
#!pip install -U accelerate

In [16]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.
/Users/nili/PycharmProjects/MasterProject/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,1.227200,1.025036,0.546634,0.482572,0.530707
2,0.934800,0.950961,0.566504,0.515010,0.560297
3,0.761800,0.969053,0.563260,0.527302,0.565425
4,0.701000,0.997964,0.568532,0.533172,0.570202


/Users/nili/PycharmProjects/MasterProject/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/nili/PycharmProjects/MasterProject/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/nili/PycharmProjects/MasterProject/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=3496, training_loss=0.8788420810306645, metrics={'train_runtime': 5457.2066, 'train_samples_per_second': 10.241, 'train_steps_per_second': 0.641, 'total_flos': 3676385621680128.0, 'train_loss': 0.8788420810306645, 'epoch': 4.0})

In [17]:
results = trainer.evaluate()

print(results)

/Users/nili/PycharmProjects/MasterProject/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.9509614706039429, 'eval_accuracy': 0.5665044606650446, 'eval_f1_macro': 0.5150100594585355, 'eval_f1_weighted': 0.5602966371115529, 'eval_runtime': 29.8899, 'eval_samples_per_second': 82.503, 'eval_steps_per_second': 5.186, 'epoch': 4.0}


In [18]:
pred = trainer.predict(val_dataset)

logits = pred.predictions
labels = pred.label_ids

probs = softmax(logits, axis=1)

predictions = np.argmax(probs, axis=1)
confidence = np.max(probs, axis=1)

true = np.argmax(labels, axis=1)

print(classification_report(true, predictions))

/Users/nili/PycharmProjects/MasterProject/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


              precision    recall  f1-score   support

           0       0.68      0.67      0.67       929
           1       0.56      0.63      0.59       463
           2       0.49      0.41      0.44       396
           3       0.63      0.25      0.36       254
           4       0.47      0.64      0.54       200
           5       0.41      0.58      0.48       224

    accuracy                           0.57      2466
   macro avg       0.54      0.53      0.52      2466
weighted avg       0.58      0.57      0.56      2466



In [19]:



import ast
from openai import OpenAI

client = OpenAI(api_key="xxx")

def query_llm(text):

    prompt = f"""
You are an expert linguistic annotator for sexism detection.

Your task is to return a probability distribution over the 6 categories below.
The probabilities MUST sum to 1.

Categories:

0: NON-SEXIST
1: IDEOLOGICAL AND INEQUALITY
2: STEREOTYPING AND DOMINANCE
3: OBJECTIFICATION
4: SEXUAL VIOLENCE
5: MISOGYNY AND NON-SEXUAL VIOLENCE

Below are annotated examples.

Example 1
Tweet: "Women belong in the kitchen."
Output:
[0.0, 0.1, 0.75, 0.05, 0.0, 0.1]

Example 2
Tweet: "Feminism is just a movement to give women unfair advantages."
Output:
[0.0, 0.8, 0.05, 0.0, 0.0, 0.15]

Example 3
Tweet: "She's only famous because she's hot."
Output:
[0.0, 0.0, 0.05, 0.8, 0.0, 0.15]

Example 4
Tweet: "I'd like to grab her like that."
Output:
[0.0, 0.0, 0.05, 0.15, 0.7, 0.1]

Example 5
Tweet: "Women are ruining society."
Output:
[0.0, 0.05, 0.05, 0.0, 0.0, 0.9]

Example 6
Tweet: "Today I had lunch with my sister and we talked about work."
Output:
[0.95, 0.01, 0.01, 0.01, 0.01, 0.01]

Now classify the following tweet.

Tweet:
{text}

Return ONLY a Python list of 6 probabilities.
Example format:
[0.1,0.2,0.1,0.3,0.2,0.1]
"""

    try:

        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )

        return ast.literal_eval(response.choices[0].message.content)

    except:
        return [1,0,0,0,0,0]

In [20]:
THRESHOLD = 0.65

texts = val_df["text"].tolist()

final_predictions = []

for i in range(len(predictions)):

    if confidence[i] >= THRESHOLD:

        final_predictions.append(predictions[i])

    else:

        llm_probs = query_llm(texts[i])

        final_predictions.append(np.argmax(llm_probs))

In [21]:
print(
    classification_report(
        true,
        final_predictions
    )
)

              precision    recall  f1-score   support

           0       0.69      0.67      0.68       929
           1       0.60      0.56      0.58       463
           2       0.53      0.28      0.37       396
           3       0.56      0.18      0.27       254
           4       0.49      0.61      0.55       200
           5       0.22      0.59      0.33       224

    accuracy                           0.52      2466
   macro avg       0.52      0.48      0.46      2466
weighted avg       0.57      0.52      0.53      2466



In [22]:
test_pred = trainer.predict(test_dataset)

test_logits = test_pred.predictions

test_probs = softmax(test_logits, axis=1)

test_preds = np.argmax(test_probs, axis=1)
test_conf = np.max(test_probs, axis=1)

/Users/nili/PycharmProjects/MasterProject/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [23]:
test_texts = test_df["text"].tolist()

final_test_predictions = []

for i in range(len(test_preds)):

    if test_conf[i] >= THRESHOLD:

        final_test_predictions.append(test_preds[i])

    else:

        llm_probs = query_llm(test_texts[i])

        final_test_predictions.append(np.argmax(llm_probs))

In [24]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "label": final_test_predictions
})

submission.to_csv("submission_soft_2.csv", index=False)